In [1]:
import subprocess, sys

def _install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

# Hugging Face Transformers is required for the pretrained Swin models
try:
    import transformers
except ImportError:
    _install('transformers')

try:
    import thop
except ImportError:
    _install('thop')

print('Dependencies ready.')

Dependencies ready.


In [2]:
import time
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

import torchvision
import torchvision.transforms as transforms

from transformers import SwinForImageClassification

try:
    from thop import profile as thop_profile
    THOP_AVAILABLE = True
except ImportError:
    THOP_AVAILABLE = False

import transformers as _tf
print(f'PyTorch      : {torch.__version__}')
print(f'Transformers : {_tf.__version__}')
print(f'CUDA         : {torch.cuda.is_available()}')

PyTorch      : 2.11.0+cu128
Transformers : 5.13.1
CUDA         : True


In [3]:
# ── Device (GPU preferred; automatic CPU fallback) ───────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

Using device: cuda


In [4]:
# ═══════════════════════════════════════════════════════════════════════════════
# Hyperparameters
# ═══════════════════════════════════════════════════════════════════════════════
NUM_CLASSES      = 100
IMG_SIZE         = 224    # Swin pretrained on 224×224; resize CIFAR for parity

# Fine-tuning hyperparameters (pretrained models)
FT_BATCH_SIZE = 32
FT_EPOCHS     = 5
FT_LR         = 2e-5

# From-scratch hyperparameters
SC_BATCH_SIZE = 32
SC_EPOCHS     = 5
SC_LR         = 1e-3

# ═══════════════════════════════════════════════════════════════════════════════
# Preprocessing
# ═══════════════════════════════════════════════════════════════════════════════
# Pretrained Swin models expect 224×224 inputs normalised with ImageNet stats.
# We apply the same pipeline to the scratch model for a fair comparison.
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

# Training transform: resize to 224, random crop + flip for regularisation
train_transform = transforms.Compose([
    transforms.Resize(256),                     # upsample 32→256
    transforms.RandomCrop(224),                 # random 224 crop
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Test transform: deterministic centre crop
test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# ── Datasets ─────────────────────────────────────────────────────────────────
train_dataset = torchvision.datasets.CIFAR100(
    root='./data', train=True,  download=True, transform=train_transform
)
test_dataset  = torchvision.datasets.CIFAR100(
    root='./data', train=False, download=True, transform=test_transform
)

_nw = 2 if torch.cuda.is_available() else 0

# Fine-tuning loaders (batch_size=32)
ft_train_loader = DataLoader(train_dataset, batch_size=FT_BATCH_SIZE,
                              shuffle=True,  num_workers=_nw,
                              pin_memory=torch.cuda.is_available())
ft_test_loader  = DataLoader(test_dataset,  batch_size=FT_BATCH_SIZE,
                              shuffle=False, num_workers=_nw,
                              pin_memory=torch.cuda.is_available())

# Scratch loaders share the same batch size and preprocessing
sc_train_loader = ft_train_loader
sc_test_loader  = ft_test_loader

print(f'Train samples : {len(train_dataset):,}')
print(f'Test  samples : {len(test_dataset):,}')
print(f'Batches/epoch : {len(ft_train_loader)}')

100%|██████████| 169M/169M [29:37<00:00, 95.1kB/s]


Train samples : 50,000
Test  samples : 10,000
Batches/epoch : 1563


In [5]:
# ═══════════════════════════════════════════════════════════════════════════════
# Compact ViT trained from scratch  (adapted from Problem 1)
# ═══════════════════════════════════════════════════════════════════════════════
# We adapt the ViT-Tiny configuration from Problem 1 to accept 224×224 inputs
# using a 16×16 patch size, giving (224/16)^2 = 196 patches.
# This is the standard ViT-Tiny recipe (Touvron et al., DeiT, 2021):
#   patch=16, emb_dim=192, num_blocks=12, num_heads=3, mlp_ratio=4
# We use a shallower variant (6 blocks) to keep training feasible in 5 epochs.


class PatchEmbedding(nn.Module):
    """Strided-conv patch projection: equivalent to a per-patch linear map."""
    def __init__(self, img_size=224, patch_size=16, in_channels=3, emb_dim=192):
        super().__init__()
        assert img_size % patch_size == 0
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, emb_dim,
                               kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)          # (B, D, H/P, W/P)
        x = x.flatten(2)          # (B, D, N)
        x = x.transpose(1, 2)     # (B, N, D)
        return x


class TransformerBlock(nn.Module):
    """Pre-LN Transformer encoder block (identical to Problem 1)."""
    def __init__(self, emb_dim, num_heads, mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(emb_dim)
        self.attn  = nn.MultiheadAttention(emb_dim, num_heads,
                                            dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(emb_dim)
        mlp_hidden = int(emb_dim * mlp_ratio)
        self.mlp   = nn.Sequential(
            nn.Linear(emb_dim, mlp_hidden), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, emb_dim), nn.Dropout(dropout),
        )

    def forward(self, x):
        n = self.norm1(x)
        x = x + self.attn(n, n, n)[0]
        x = x + self.mlp(self.norm2(x))
        return x


class ScratchViT(nn.Module):
    """
    Compact Vision Transformer trained from random initialisation.

    Config: img=224, patch=16, emb=192, blocks=6, heads=3, mlp_ratio=4.
    This mirrors the ViT-Tiny recipe and serves as the from-scratch baseline
    against the pretrained Swin Transformers.
    """
    def __init__(self, img_size=224, patch_size=16, in_channels=3,
                 num_classes=100, emb_dim=192, num_blocks=6,
                 num_heads=3, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, emb_dim)
        N = self.patch_embed.num_patches
        self.cls_token = nn.Parameter(torch.zeros(1, 1, emb_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, N + 1, emb_dim))
        self.pos_drop  = nn.Dropout(dropout)
        self.blocks    = nn.Sequential(*[
            TransformerBlock(emb_dim, num_heads, mlp_ratio, dropout)
            for _ in range(num_blocks)
        ])
        self.norm = nn.LayerNorm(emb_dim)
        self.head = nn.Linear(emb_dim, num_classes)
        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out')

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)
        x = torch.cat([self.cls_token.expand(B, -1, -1), x], dim=1)
        x = self.pos_drop(x + self.pos_embed)
        x = self.blocks(x)
        return self.head(self.norm(x)[:, 0])


# Sanity check
_v = ScratchViT()
assert _v(torch.randn(2, 3, 224, 224)).shape == (2, 100)
print('ScratchViT forward: OK')
del _v

ScratchViT forward: OK


In [6]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def count_all_parameters(model):
    """Total params (including frozen backbone) for size reporting."""
    return sum(p.numel() for p in model.parameters())

def get_flops(model, img_size=224):
    if not THOP_AVAILABLE:
        return None
    dummy = torch.randn(1, 3, img_size, img_size).to(next(model.parameters()).device)
    try:
        macs, _ = thop_profile(model, inputs=(dummy,), verbose=False)
        return int(macs * 2)
    except Exception:
        return None

In [7]:
# ═══════════════════════════════════════════════════════════════════════════════
# Training helpers — unified interface for HuggingFace and plain PyTorch models
# ═══════════════════════════════════════════════════════════════════════════════


def train_one_epoch_hf(model, loader, optimizer, criterion):
    """One epoch for a HuggingFace SwinForImageClassification model."""
    model.train()
    total_loss = correct = total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        # Pass pixel_values; let the model compute its own loss
        outputs = model(pixel_values=images, labels=labels)
        loss    = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        correct    += (outputs.logits.argmax(1) == labels).sum().item()
        total      += images.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate_hf(model, loader):
    """Evaluation for HuggingFace model."""
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss = correct = total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(pixel_values=images)
        loss    = criterion(outputs.logits, labels)
        total_loss += loss.item() * images.size(0)
        correct    += (outputs.logits.argmax(1) == labels).sum().item()
        total      += images.size(0)

    return total_loss / total, correct / total


def train_one_epoch_pt(model, loader, optimizer, criterion):
    """One epoch for a plain PyTorch model."""
    model.train()
    total_loss = correct = total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += images.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate_pt(model, loader, criterion):
    """Evaluation for plain PyTorch model."""
    model.eval()
    total_loss = correct = total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss   = criterion(logits, labels)
        total_loss += loss.item() * images.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += images.size(0)

    return total_loss / total, correct / total


def run_finetuning(model_name, hf_model_id, train_loader, test_loader,
                   num_epochs=FT_EPOCHS, lr=FT_LR):
    """
    Load a pretrained Swin from Hugging Face, replace the head for 100 classes,
    freeze the backbone, and fine-tune the head only.

    Returns a results dict.
    """
    print(f'\n{"="*68}')
    print(f'  Loading : {hf_model_id}')

    # Load pretrained weights; replace the classification head automatically
    model = SwinForImageClassification.from_pretrained(
        hf_model_id,
        num_labels       = NUM_CLASSES,
        ignore_mismatched_sizes = True,  # head size changes 1000→100
    )

    # Freeze all backbone parameters — only the linear head is trainable
    for param in model.swin.parameters():
        param.requires_grad = False

    model = model.to(device)

    trainable = count_parameters(model)
    total     = count_all_parameters(model)
    flops     = get_flops(model)

    print(f'  Trainable params : {trainable:,}  (head only)')
    print(f'  Total params     : {total:,}')
    if flops:
        print(f'  FLOPs/img        : {flops/1e9:.2f} G')
    print(f'{"="*68}')

    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=lr
    )

    epoch_times  = []
    train_losses, train_accs = [], []
    test_losses,  test_accs  = [], []
    wall_start   = time.time()

    for epoch in range(1, num_epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch_hf(model, train_loader, optimizer, None)
        te_loss, te_acc = evaluate_hf(model, test_loader)
        elapsed = time.time() - t0

        epoch_times.append(elapsed)
        train_losses.append(tr_loss); train_accs.append(tr_acc)
        test_losses.append(te_loss);  test_accs.append(te_acc)

        print(f'  Epoch {epoch}/{num_epochs} | '
              f'Train loss={tr_loss:.4f} acc={tr_acc*100:.2f}% | '
              f'Test  loss={te_loss:.4f} acc={te_acc*100:.2f}% | {elapsed:.1f}s')

    total_time = time.time() - wall_start
    print(f'  Final test acc : {test_accs[-1]*100:.2f}%  |  Total: {total_time:.1f}s')

    return {
        'name'            : model_name,
        'trainable_params': trainable,
        'total_params'    : total,
        'flops'           : flops,
        'train_losses'    : train_losses,
        'train_accs'      : train_accs,
        'test_losses'     : test_losses,
        'test_accs'       : test_accs,
        'epoch_times'     : epoch_times,
        'total_time_sec'  : total_time,
        'mean_epoch_sec'  : float(np.mean(epoch_times)),
        'final_test_acc'  : test_accs[-1],
        'final_train_loss': train_losses[-1],
        'model'           : model,
    }


def run_scratch(model, model_name, train_loader, test_loader,
                num_epochs=SC_EPOCHS, lr=SC_LR):
    """Train a plain PyTorch model from random initialisation."""
    model = model.to(device)
    trainable = count_parameters(model)
    total     = count_all_parameters(model)
    flops     = get_flops(model)

    print(f'\n{"="*68}')
    print(f'  Model            : {model_name}')
    print(f'  Trainable params : {trainable:,}')
    print(f'  Total params     : {total:,}')
    if flops:
        print(f'  FLOPs/img        : {flops/1e9:.2f} G')
    print(f'{"="*68}')

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    epoch_times  = []
    train_losses, train_accs = [], []
    test_losses,  test_accs  = [], []
    wall_start   = time.time()

    for epoch in range(1, num_epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch_pt(model, train_loader, optimizer, criterion)
        te_loss, te_acc = evaluate_pt(model, test_loader, criterion)
        elapsed = time.time() - t0

        epoch_times.append(elapsed)
        train_losses.append(tr_loss); train_accs.append(tr_acc)
        test_losses.append(te_loss);  test_accs.append(te_acc)

        print(f'  Epoch {epoch}/{num_epochs} | '
              f'Train loss={tr_loss:.4f} acc={tr_acc*100:.2f}% | '
              f'Test  loss={te_loss:.4f} acc={te_acc*100:.2f}% | {elapsed:.1f}s')

    total_time = time.time() - wall_start
    print(f'  Final test acc : {test_accs[-1]*100:.2f}%  |  Total: {total_time:.1f}s')

    return {
        'name'            : model_name,
        'trainable_params': trainable,
        'total_params'    : total,
        'flops'           : flops,
        'train_losses'    : train_losses,
        'train_accs'      : train_accs,
        'test_losses'     : test_losses,
        'test_accs'       : test_accs,
        'epoch_times'     : epoch_times,
        'total_time_sec'  : total_time,
        'mean_epoch_sec'  : float(np.mean(epoch_times)),
        'final_test_acc'  : test_accs[-1],
        'final_train_loss': train_losses[-1],
        'model'           : model,
    }

In [8]:
# ═══════════════════════════════════════════════════════════════════════════════
# Fine-tune Swin-Tiny (pretrained on ImageNet-1K, 28 M params)
# ═══════════════════════════════════════════════════════════════════════════════

swin_tiny_result = run_finetuning(
    model_name  = 'Swin-Tiny (pretrained, head-only)',
    hf_model_id = 'microsoft/swin-tiny-patch4-window7-224',
    train_loader= ft_train_loader,
    test_loader = ft_test_loader,
    num_epochs  = FT_EPOCHS,
    lr          = FT_LR,
)


  Loading : microsoft/swin-tiny-patch4-window7-224


config.json:   0%|          | 0.00/71.8k [00:00<?, ?B/s]

[transformers] You passed `num_labels=100` which is incompatible to the `id2label` map of length `1000`.


model.safetensors: reconstructing file:   0%|          |  0.00B /  113MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/221 [00:00<?, ?it/s]

[transformers] SwinForImageClassification LOAD REPORT from: microsoft/swin-tiny-patch4-window7-224
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([100, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([100])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


  Trainable params : 76,900  (head only)
  Total params     : 27,596,254
  FLOPs/img        : 8.73 G
  Epoch 1/5 | Train loss=4.0783 acc=22.46% | Test  loss=3.5316 acc=44.87% | 288.2s
  Epoch 2/5 | Train loss=3.1039 acc=52.41% | Test  loss=2.7127 acc=56.85% | 300.9s
  Epoch 3/5 | Train loss=2.4326 acc=59.30% | Test  loss=2.1730 acc=61.04% | 301.3s
  Epoch 4/5 | Train loss=2.0037 acc=62.40% | Test  loss=1.8351 acc=63.38% | 301.7s
  Epoch 5/5 | Train loss=1.7336 acc=64.59% | Test  loss=1.6210 acc=65.21% | 300.1s
  Final test acc : 65.21%  |  Total: 1492.2s


In [9]:
# ═══════════════════════════════════════════════════════════════════════════════
# Fine-tune Swin-Small (pretrained on ImageNet-1K, 50 M params)
# ═══════════════════════════════════════════════════════════════════════════════
# Swin-Small uses the same patch embedding and window attention as Swin-Tiny
# but doubles the channel width at each stage, giving a larger feature space
# at the cost of ~2× more parameters and FLOPs.

swin_small_result = run_finetuning(
    model_name  = 'Swin-Small (pretrained, head-only)',
    hf_model_id = 'microsoft/swin-small-patch4-window7-224',
    train_loader= ft_train_loader,
    test_loader = ft_test_loader,
    num_epochs  = FT_EPOCHS,
    lr          = FT_LR,
)


  Loading : microsoft/swin-small-patch4-window7-224


config.json:   0%|          | 0.00/71.8k [00:00<?, ?B/s]

[transformers] You passed `num_labels=100` which is incompatible to the `id2label` map of length `1000`.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  199MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

...

[transformers] SwinForImageClassification LOAD REPORT from: microsoft/swin-small-patch4-window7-224
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([100, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([100])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


  Trainable params : 76,900  (head only)
  Total params     : 48,914,158
  FLOPs/img        : 17.07 G


model.safetensors: reconstructing file:   0%|          |  0.00B /  199MB            

model.safetensors: downloading bytes:           |  0.00B            

  Epoch 1/5 | Train loss=4.0015 acc=26.58% | Test  loss=3.4118 acc=49.16% | 530.8s
  Epoch 2/5 | Train loss=2.9383 acc=55.86% | Test  loss=2.5310 acc=60.76% | 530.3s
  Epoch 3/5 | Train loss=2.2226 acc=62.85% | Test  loss=1.9728 acc=65.04% | 530.1s
  Epoch 4/5 | Train loss=1.7925 acc=65.52% | Test  loss=1.6451 acc=67.37% | 530.4s
  Epoch 5/5 | Train loss=1.5351 acc=67.50% | Test  loss=1.4468 acc=68.99% | 530.8s
  Final test acc : 68.99%  |  Total: 2652.3s


In [10]:
# ═══════════════════════════════════════════════════════════════════════════════
# Train ViT-Tiny from scratch (random initialisation)
# ═══════════════════════════════════════════════════════════════════════════════
# Config: img=224, patch=16, emb=192, blocks=6, heads=3, mlp_ratio=4
# Same preprocessing as the pretrained Swin models (224×224, ImageNet stats).
# LR=0.001 (higher than fine-tuning because weights are random).

scratch_vit = ScratchViT(
    img_size    = 224,
    patch_size  = 16,
    in_channels = 3,
    num_classes = NUM_CLASSES,
    emb_dim     = 192,
    num_blocks  = 6,
    num_heads   = 3,
    mlp_ratio   = 4.0,
    dropout     = 0.1,
)

scratch_result = run_scratch(
    model        = scratch_vit,
    model_name   = 'ViT-Tiny (from scratch)',
    train_loader = sc_train_loader,
    test_loader  = sc_test_loader,
    num_epochs   = SC_EPOCHS,
    lr           = SC_LR,
)


  Model            : ViT-Tiny (from scratch)
  Trainable params : 2,874,532
  Total params     : 2,874,532
  FLOPs/img        : 0.76 G
  Epoch 1/5 | Train loss=3.9850 acc=8.20% | Test  loss=3.7477 acc=12.07% | 148.5s
  Epoch 2/5 | Train loss=3.6413 acc=13.59% | Test  loss=3.5484 acc=15.17% | 147.3s
  Epoch 3/5 | Train loss=3.4398 acc=17.09% | Test  loss=3.3263 acc=19.74% | 146.7s
  Epoch 4/5 | Train loss=3.2608 acc=20.28% | Test  loss=3.1755 acc=22.18% | 148.3s
  Epoch 5/5 | Train loss=3.1018 acc=23.40% | Test  loss=3.0490 acc=24.69% | 147.5s
  Final test acc : 24.69%  |  Total: 738.3s


In [11]:
# ═══════════════════════════════════════════════════════════════════════════════
# Summary results table
# ═══════════════════════════════════════════════════════════════════════════════

all_results = [swin_tiny_result, swin_small_result, scratch_result]

print('\nFinal Results — CIFAR-100 (5 epochs, batch=32)')
print('=' * 100)
print(
    f'{"Model":<38} {"Total Params":>12} {"Trainable":>12} '
    f'{"FLOPs":>8} {"Test Acc":>9} {"Ep (s)":>7} {"Total (s)":>9}'
)
print('-' * 100)

for r in all_results:
    f_str = f'{r["flops"]/1e9:.2f}G' if r['flops'] else '—'
    print(
        f'{r["name"]:<38} {r["total_params"]:>12,} {r["trainable_params"]:>12,} '
        f'{f_str:>8} {r["final_test_acc"]*100:>8.2f}% '
        f'{r["mean_epoch_sec"]:>7.1f} {r["total_time_sec"]:>9.1f}'
    )

print()

# Per-epoch test accuracy
print('\nTest Accuracy (%) per Epoch')
labels = [r['name'][:22] for r in all_results]
print(f'{"Epoch":<7}', '  '.join(f'{l:>22}' for l in labels))
print('-' * (7 + 25 * len(all_results)))
for ep in range(FT_EPOCHS):
    row = f'{ep+1:<7}'
    for r in all_results:
        row += f'  {r["test_accs"][ep]*100:>22.2f}'
    print(row)


Final Results — CIFAR-100 (5 epochs, batch=32)
Model                                  Total Params    Trainable    FLOPs  Test Acc  Ep (s) Total (s)
----------------------------------------------------------------------------------------------------
Swin-Tiny (pretrained, head-only)        27,596,254       76,900    8.73G    65.21%   298.4    1492.2
Swin-Small (pretrained, head-only)       48,914,158       76,900   17.07G    68.99%   530.5    2652.3
ViT-Tiny (from scratch)                   2,874,532    2,874,532    0.76G    24.69%   147.7     738.3


Test Accuracy (%) per Epoch
Epoch   Swin-Tiny (pretrained,  Swin-Small (pretrained  ViT-Tiny (from scratch
----------------------------------------------------------------------------------
1                         44.87                   49.16                   12.07
2                         56.85                   60.76                   15.17
3                         61.04                   65.04                   19.74
4          